# **TF IDF & CBOW**

## **Load Data**

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, sys, time
from google.colab import drive
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
pta_trunojoyo_path = "/content/drive/MyDrive/PPW/output/pta_manajemen.csv"
pta_trunojoyo = pd.read_csv(pta_trunojoyo_path)

pta_trunojoyo


,id,penulis,judul,abstrak_id,pembimbing_pertama,pembimbing_kedua,prodi
0,80211100070,SATIYAH,PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...","Dra. Hj. S. Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST.SE,M.MT",Manajemen
1,90211200001,Faishal,ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...,Tujuan penelitian ini adalah untuk mengetahui ...,Nurita Andriani,Yustina Chrismardani,Manajemen
2,80211100050,Wahyu Kurniawan,PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...,NaN,"Dr. Dra. Hj. Iriani Ismail, MM","Dra. Hj. S. Anugrahini Irawati, MM",Manajemen
3,100211200002,Muhammad Zakaria Utomo,Pengukuran Website Quality Pada Situs Sistem A...,Aplikasi nyata pemanfaatan teknologi informasi...,"Dr. Ir. Nurita Andriani, MM","Nirma Kurriwati, SP, M.Si",Manajemen
4,80211100044,Hendri Wahyudi Prayitno,PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...,Abstrak\r\nPenelitian ini menggunakan metode k...,"Dra. Hj. S Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST,SE,.MT",Manajemen
...,...,...,...,...,...,...,...
1026,160211100071,Husnul Hotimah,Analisis Cost Volume Profit Untuk Menentukan T...,ABSTRAK\nPenelitian ini bertujuan untuk menget...,"Hj. Evaliati Amaniyah, S.E., M.S.M.",NaN,Manajemen
1027,160211100291,Uswatun Hasanah,Pengaruh Pelatihan Dan Kompensasi Terhadap Pro...,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar...","Dr. Raden Mas Mochammad Wispandono S.E ., MS",NaN,Manajemen
1028,160211100064,ACH FATHONI,PERAN SERVICE PERFORMANCE DAN CLIMATE ORGANIZA...,ABSTRAK\nTujuan dari penelitian ini adalah unt...,"YUDHI PRASETYA MADA, S.E., M.M.",NaN,Manajemen
1029,160211100030,INTAN YULLIA NINGSIH,BAURAN PROMOSI PADA DEALER YAMAHA TRETAN MOTOR...,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...,"DR. MOHAMMAD ARIEF, S.E., M.M.",NaN,Manajemen


## **Mengecek Nilai yang null dan memebersihkannya**


In [6]:
# Cek jumlah NaN di tiap kolom
print(pta_trunojoyo.isna().sum())


id                     0
penulis                2
judul                  0
abstrak_id             5
pembimbing_pertama     3
pembimbing_kedua      94
prodi                  0
dtype: int64


In [7]:
# Hapus baris yang abstrak_en kosong
df = pta_trunojoyo.dropna(subset=['abstrak_id']).reset_index(drop=True)

# Cek lagi apakah masih ada yang kosong
print(df['abstrak_id'].isna().sum())


0


In [8]:
df

,id,penulis,judul,abstrak_id,pembimbing_pertama,pembimbing_kedua,prodi
0,80211100070,SATIYAH,PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...","Dra. Hj. S. Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST.SE,M.MT",Manajemen
1,90211200001,Faishal,ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...,Tujuan penelitian ini adalah untuk mengetahui ...,Nurita Andriani,Yustina Chrismardani,Manajemen
2,100211200002,Muhammad Zakaria Utomo,Pengukuran Website Quality Pada Situs Sistem A...,Aplikasi nyata pemanfaatan teknologi informasi...,"Dr. Ir. Nurita Andriani, MM","Nirma Kurriwati, SP, M.Si",Manajemen
3,80211100044,Hendri Wahyudi Prayitno,PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...,Abstrak\r\nPenelitian ini menggunakan metode k...,"Dra. Hj. S Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST,SE,.MT",Manajemen
4,80211100119,Aththaariq,PENGARUH KOMPETENSI DOSEN TERHADAP KINERJA DOS...,"Abstrak\r\n\r\nAththaariq, Pengaruh Kompetensi...","Dr.RM Moch Wispandono,.S.E,.MS","Dr. Muhammad Alkirom Wildan,S.E.,M.Si.",Manajemen
...,...,...,...,...,...,...,...
1021,160211100071,Husnul Hotimah,Analisis Cost Volume Profit Untuk Menentukan T...,ABSTRAK\nPenelitian ini bertujuan untuk menget...,"Hj. Evaliati Amaniyah, S.E., M.S.M.",NaN,Manajemen
1022,160211100291,Uswatun Hasanah,Pengaruh Pelatihan Dan Kompensasi Terhadap Pro...,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar...","Dr. Raden Mas Mochammad Wispandono S.E ., MS",NaN,Manajemen
1023,160211100064,ACH FATHONI,PERAN SERVICE PERFORMANCE DAN CLIMATE ORGANIZA...,ABSTRAK\nTujuan dari penelitian ini adalah unt...,"YUDHI PRASETYA MADA, S.E., M.M.",NaN,Manajemen
1024,160211100030,INTAN YULLIA NINGSIH,BAURAN PROMOSI PADA DEALER YAMAHA TRETAN MOTOR...,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...,"DR. MOHAMMAD ARIEF, S.E., M.M.",NaN,Manajemen


## **Cleaning Data**

Kode tersebut melakukan pembersihan teks pada kolom abstrak_indo dengan mengubah semua huruf menjadi huruf kecil agar konsisten dan tidak membedakan antara huruf besar dan kecil. Selain itu, kode ini menghapus semua angka, tanda baca, dan simbol sehingga hanya tersisa huruf dan spasi, yang bertujuan menghilangkan karakter yang tidak relevan atau mengganggu proses analisis teks. Terakhir, pembersihan juga menghapus spasi berlebihan, termasuk spasi ganda atau spasi di awal dan akhir kalimat, agar teks menjadi lebih rapi dan mudah diproses. Proses ini penting untuk menyederhanakan dan menormalkan data teks sehingga siap digunakan dalam tahap analisis lebih lanjut seperti tokenisasi, stemming, atau klasifikasi.

In [10]:

import pandas as pd
import re

# Fungsi cleansing
def cleansing(text):
    text = str(text).lower()                         # ubah ke huruf kecil
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)        # hilangkan angka/simbol
    text = re.sub(r'\s+', ' ', text).strip()        # hilangkan spasi berlebih
    return text

# Terapkan cleansing
df['cleaned'] = df['abstrak_id'].apply(cleansing)

# Simpan ke CSV
df.to_csv('/content/drive/MyDrive/PPW/output/manajemen_abstrak_cleaned.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
df[['abstrak_id','cleaned']].head()

Data berhasil disimpan ke CSV!



,abstrak_id,cleaned
0,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...",abstrak satiyah pengaruh faktor faktor pelatih...
1,Tujuan penelitian ini adalah untuk mengetahui ...,tujuan penelitian ini adalah untuk mengetahui ...
2,Aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata pemanfaatan teknologi informasi...
3,Abstrak\r\nPenelitian ini menggunakan metode k...,abstrak penelitian ini menggunakan metode kuan...
4,"Abstrak\r\n\r\nAththaariq, Pengaruh Kompetensi...",abstrak aththaariq pengaruh kompetensi dosen t...


In [11]:
!pip install pyspellchecker

## **Stopword Removal**
Kode ini digunakan untuk menghapus kata-kata umum (stopwords) dalam bahasa Indonesia dari teks yang sudah dibersihkan di kolom cleaned. Dengan menggunakan library Sastrawi, dibuat objek stopword remover yang akan menghilangkan kata-kata seperti “dan”, “di”, “yang”, dan kata umum lain yang biasanya tidak membawa makna penting dalam analisis teks.

Setelah proses penghilangan stopwords selesai, hasilnya disimpan dalam kolom baru bernama no_stopwords. Kemudian, data yang sudah diproses ini disimpan kembali ke file CSV baru manajemen_abstrak_no_stopwords.csv. Langkah ini penting agar teks menjadi lebih fokus pada kata-kata bermakna dan memudahkan analisis selanjutnya seperti klasifikasi atau clustering.

In [12]:
!pip install Sastrawi

In [13]:
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Buat stopword remover
factory = StopWordRemoverFactory()
stopword = factory.create_stop_word_remover()

# Terapkan stopword removal pada kolom 'cleaned'
df['no_stopwords'] = df['cleaned'].apply(stopword.remove)

# Simpan hasil ke CSV
df.to_csv('/content/drive/MyDrive/PPW/output/manajemen_abstrak_no_stopwords.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
df[['cleaned','no_stopwords']].head()


Data berhasil disimpan ke CSV!



,cleaned,no_stopwords
0,abstrak satiyah pengaruh faktor faktor pelatih...,abstrak satiyah pengaruh faktor faktor pelatih...
1,tujuan penelitian ini adalah untuk mengetahui ...,tujuan penelitian mengetahui persepsi brand as...
2,aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata pemanfaatan teknologi informasi...
3,abstrak penelitian ini menggunakan metode kuan...,abstrak penelitian menggunakan metode kuantita...
4,abstrak aththaariq pengaruh kompetensi dosen t...,abstrak aththaariq pengaruh kompetensi dosen k...


## **Stemming**
Kode ini melakukan proses stemming pada teks di kolom no_stopwords menggunakan library Sastrawi, yaitu mengubah kata-kata yang memiliki imbuhan atau variasi menjadi bentuk dasar atau kata dasarnya. Hasil stemming ini disimpan dalam kolom baru stemmed dan kemudian disimpan ke file CSV baru untuk digunakan dalam analisis selanjutnya. Proses stemming penting untuk menyederhanakan variasi kata sehingga model atau analisis teks dapat mengenali kata-kata dengan makna yang sama secara konsisten.

In [14]:
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Buat stemmer
stemmer = StemmerFactory().create_stemmer()

# Terapkan stemming pada kolom 'no_stopwords'
df['stemmed'] = df['no_stopwords'].apply(stemmer.stem)

# Simpan hasil ke CSV
df.to_csv('/content/drive/MyDrive/PPW/output/manajemen_abstrak_stemmed.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
df[['no_stopwords','stemmed']].head()


Data berhasil disimpan ke CSV!



,no_stopwords,stemmed
0,abstrak satiyah pengaruh faktor faktor pelatih...,abstrak satiyah pengaruh faktor faktor latih k...
1,tujuan penelitian mengetahui persepsi brand as...,tuju teliti tahu persepsi brand association la...
2,aplikasi nyata pemanfaatan teknologi informasi...,aplikasi nyata manfaat teknologi informasi kom...
3,abstrak penelitian menggunakan metode kuantita...,abstrak teliti guna metode kuantitatif tekan u...
4,abstrak aththaariq pengaruh kompetensi dosen k...,abstrak aththaariq pengaruh kompetensi dosen k...


## **Tokenisasi**
Kode ini melakukan tokenisasi pada teks yang sudah melalui proses stemming di kolom stemmed. Tokenisasi adalah proses memecah kalimat atau teks menjadi potongan-potongan kata (token) yang lebih kecil, biasanya berdasarkan spasi. Fungsi tokenize yang sederhana di sini memecah setiap kalimat menjadi daftar kata-kata dengan menggunakan metode split(). Hasil tokenisasi disimpan dalam kolom baru tokens, yang berisi daftar kata untuk tiap baris teks. Setelah itu, data lengkap dengan token disimpan ke file CSV baru manajemen_abstrak_tokens.csv

In [15]:
import pandas as pd

# Fungsi tokenisasi sederhana
def tokenize(text):
    return text.split()

# Terapkan tokenisasi pada kolom 'stemmed'
df['tokens'] = df['stemmed'].apply(tokenize)

# Simpan hasil ke CSV
df.to_csv('/content/drive/MyDrive/PPW/output/manajemen_abstrak_tokens.csv', index=False)
print("Data berhasil disimpan ke CSV!\n")

# Tampilkan beberapa baris pertama
df[['stemmed','tokens']].head()


Data berhasil disimpan ke CSV!



,stemmed,tokens
0,abstrak satiyah pengaruh faktor faktor latih k...,"[abstrak, satiyah, pengaruh, faktor, faktor, l..."
1,tuju teliti tahu persepsi brand association la...,"[tuju, teliti, tahu, persepsi, brand, associat..."
2,aplikasi nyata manfaat teknologi informasi kom...,"[aplikasi, nyata, manfaat, teknologi, informas..."
3,abstrak teliti guna metode kuantitatif tekan u...,"[abstrak, teliti, guna, metode, kuantitatif, t..."
4,abstrak aththaariq pengaruh kompetensi dosen k...,"[abstrak, aththaariq, pengaruh, kompetensi, do..."


## **TF-IDF**

### **Ambil Corpus**

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Ambil corpus dari kolom stemmed
corpus = df['stemmed'].astype(str).tolist()

print("Jumlah dokumen dalam corpus:", len(corpus))
print("Contoh dokumen:", corpus[0][:200])


Jumlah dokumen dalam corpus: 1026
Contoh dokumen: abstrak satiyah pengaruh faktor faktor latih kembang produktivitas kerja dinas laut ikan bangkal bawah bimbing dra hj s anugrahini irawati mm helm buyung aulia s st se m mt upaya tingkat produktivitas


In [17]:
# Inisialisasi TF-IDF
vectorizer = TfidfVectorizer()

# Fit dan transform
X_tfidf = vectorizer.fit_transform(corpus)

print("Shape TF-IDF:", X_tfidf.shape)  # (jumlah dokumen, jumlah kata unik)


Shape TF-IDF: (1026, 6056)


In [18]:
# Ubah ke DataFrame biar mudah dibaca
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Tampilkan 5 baris pertama
tfidf_df.head()


,aaa,aaaamanahsyariah,aar,abadi,abai,abalisis,abc,abcs,abd,abdul,...,zscore,zte,zuhri,zuhruf,zulfi,zulia,zuliana,zulkifli,zulpah,zyn
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
# Tampilkan 10 kata dengan bobot TF-IDF tertinggi di dokumen pertama
print(tfidf_df.iloc[0].sort_values(ascending=False).head(10))

produktivitas    0.411053
laut             0.331714
ikan             0.331714
latih            0.328843
dinas            0.234956
kembang          0.223807
pegawai          0.223024
faktor           0.222064
seleksi          0.165857
kerja            0.158086
Name: 0, dtype: float64


## **CBOW**

In [1]:
!pip install gensim


In [20]:
from gensim.models import Word2Vec

corpus = []
for col in df['stemmed']:
    word_list = col.split(" ")
    corpus.append(word_list)

# Tampilkan contoh dokumen pertama
print(corpus[0][:20])


['abstrak', 'satiyah', 'pengaruh', 'faktor', 'faktor', 'latih', 'kembang', 'produktivitas', 'kerja', 'dinas', 'laut', 'ikan', 'bangkal', 'bawah', 'bimbing', 'dra', 'hj', 's', 'anugrahini', 'irawati']


In [21]:
# Training Word2Vec
model = Word2Vec(
    corpus,
    vector_size=56,   # ukuran embedding
    window=5,
    min_count=1,
    sg=0   # CBOW
)


In [23]:
import numpy as np

class MeanEmbeddingVectorizer:
    def __init__(self, model):
        self.model = model
        self.vector_size = model.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.array([
            np.mean([self.model.wv[word] for word in words if word in self.model.wv]
                    or [np.zeros(self.vector_size)], axis=0)
            for words in X
        ])

    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)


In [24]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(corpus)

# simpan ke DataFrame
df['array'] = list(mean_embedded)

# cek panjang embedding
df['embedding_length'] = df['array'].str.len()
print(df[['stemmed','embedding_length']].head())


                                             stemmed  embedding_length
0  abstrak satiyah pengaruh faktor faktor latih k...                56
1  tuju teliti tahu persepsi brand association la...                56
2  aplikasi nyata manfaat teknologi informasi kom...                56
3  abstrak teliti guna metode kuantitatif tekan u...                56
4  abstrak aththaariq pengaruh kompetensi dosen k...                56


In [25]:
num_features = len(df['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris di kolom 'embedding'
for embedding_list in df['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_df = pd.DataFrame(data_dict)

print(embedding_df)

            f1        f2        f3        f4        f5        f6        f7  \
0    -0.551324  0.388220  0.053571  0.105566  0.338856 -0.632777  0.729009   
1    -0.371729  0.697712  0.238175 -0.245901  0.106839 -0.615433  0.583872   
2    -0.371125  0.468522  0.128401 -0.015336  0.227745 -0.550824  0.393297   
3    -0.456337  1.041008  0.267175  0.502781  0.777465 -0.313021  1.269311   
4    -0.447965  0.762816  0.211767  0.209391  0.389642 -0.455004  0.891643   
...        ...       ...       ...       ...       ...       ...       ...   
1021 -0.107750  0.644889 -0.109320  0.241382  0.397885 -0.079862  0.626408   
1022 -0.461898  0.783063  0.015679  0.796692  1.093387 -0.200978  1.398537   
1023 -0.337914  0.933209  0.365587  0.380016  0.636457 -0.329424  0.984034   
1024 -0.577772  1.091340  0.329223 -0.152980  0.085724 -0.463016  0.364783   
1025 -0.589364  0.370781  0.077256  0.164916  0.478848 -0.634359  0.661505   

            f8        f9       f10  ...       f47       f48    

In [26]:
embedding_df['abstrak_id'] = df['abstrak_id'].values

In [27]:
embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f48,f49,f50,f51,f52,f53,f54,f55,f56,abstrak_id
0,-0.551324,0.388220,0.053571,0.105566,0.338856,-0.632777,0.729009,-0.411126,-0.219308,-0.406394,...,0.983531,-0.016429,0.420148,-0.057519,0.997285,-0.349098,-0.216298,0.460129,-0.249733,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel..."
1,-0.371729,0.697712,0.238175,-0.245901,0.106839,-0.615433,0.583872,-0.431262,-0.067639,-0.431545,...,0.152169,-0.042778,0.256450,-0.218609,0.805347,-0.449102,0.072762,0.482869,-0.144076,Tujuan penelitian ini adalah untuk mengetahui ...
2,-0.371125,0.468522,0.128401,-0.015336,0.227745,-0.550824,0.393297,-0.335645,-0.097965,-0.384134,...,0.351191,-0.005439,0.215160,-0.224152,0.747748,-0.281760,-0.100098,0.542154,-0.094177,Aplikasi nyata pemanfaatan teknologi informasi...
3,-0.456337,1.041008,0.267175,0.502781,0.777465,-0.313021,1.269311,-0.407867,-0.242763,-0.659517,...,1.105825,-0.255982,0.516595,0.005752,1.491357,-0.332713,-0.241774,0.449496,-0.256919,Abstrak\r\nPenelitian ini menggunakan metode k...
4,-0.447965,0.762816,0.211767,0.209391,0.389642,-0.455004,0.891643,-0.557078,-0.263596,-0.636704,...,0.751044,-0.234562,0.355774,-0.162276,1.149517,-0.364969,-0.114973,0.483569,-0.322280,"Abstrak\r\n\r\nAththaariq, Pengaruh Kompetensi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1021,-0.107750,0.644889,-0.109320,0.241382,0.397885,-0.079862,0.626408,-0.761884,-0.124874,-0.636867,...,0.331126,-0.288512,0.105752,0.066841,0.926425,-0.306657,0.189142,0.157148,0.164842,ABSTRAK\nPenelitian ini bertujuan untuk menget...
1022,-0.461898,0.783063,0.015679,0.796692,1.093387,-0.200978,1.398537,-0.748345,-0.463825,-0.750832,...,1.480213,-0.261435,0.334038,0.315427,1.704503,-0.142412,-0.276465,0.297869,-0.296608,"ABSTRAK\nUswatun Hasanah, 160211100291, Pengar..."
1023,-0.337914,0.933209,0.365587,0.380016,0.636457,-0.329424,0.984034,-0.546755,-0.230430,-0.751430,...,0.518240,-0.229606,0.294014,-0.021526,1.254922,-0.313612,-0.024855,0.372219,-0.062571,ABSTRAK\nTujuan dari penelitian ini adalah unt...
1024,-0.577772,1.091340,0.329223,-0.152980,0.085724,-0.463016,0.364783,-0.506945,0.005791,-0.701045,...,0.887270,-0.169765,0.238389,-0.180052,0.889018,-0.244645,-0.155732,0.650621,-0.429813,ABSTRAK\nPenelitian ini bertujuan: (1) Untuk m...


In [28]:
embedding_df.shape

(1026, 57)